# Парсер и EDA построения модели предсказания риска раннего увольнения

Colab-ноутбук для ВКР: сбор публичных резюме, деидентификация, полноценный EDA, label-free модель риска раннего увольнения и диагностические метрики модели.

**Что делает ноутбук:**

1. Создает внутри Colab модули парсера и модели.
2. Парсит публичные HTML-страницы резюме.
3. Сохраняет JSONL/CSV без прямых персональных данных.
4. Строит EDA по качеству данных, зарплате, опыту, городам, отраслям и карьерной стабильности.
5. Считает proxy-risk.
6. Визуализирует метрики: coverage, confidence, sanity correlations, bootstrap stability, decile-анализ и компоненты риска.

**Важно:** Результат является интерпретируемым прокси-риском по карьерной траектории, а не доказанным фактом будущего увольнения.

## Quick Start Для Colab

1. Выполните ячейку setup.
2. При необходимости включите `USE_GOOGLE_DRIVE = True`, чтобы результаты сохранялись в Google Drive.
3. Оставьте `MAX_RESUMES = 20000` для статистически осмысленного EDA.
4. Запустите парсинг. Полный прогон на 20000 резюме обычно занимает очень длительное время из-за уважительной задержки между запросами.
5. Выполните блоки EDA, scoring и metrics.

Если Colab-сессия оборвалась, поставьте `RUN_PARSING = False` и продолжайте с уже сохраненного `data/resumes.jsonl`.

In [10]:
# Colab setup
!pip -q install pandas matplotlib seaborn tqdm

from pathlib import Path
import json
import math
import time
import csv
import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm


# Включите True, если хотите сохранять JSONL/CSV и метрики в Google Drive.
USE_GOOGLE_DRIVE = False
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/retention')

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_DIR = DRIVE_PROJECT_DIR
    except Exception as exc:
        print('Google Drive mount failed, falling back to local Colab storage:', exc)
        PROJECT_DIR = Path('.')
else:
    PROJECT_DIR = Path('.')

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROJECT_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 140)
sns.set_theme(style='whitegrid', context='notebook')

print('Project dir:', PROJECT_DIR.resolve())
print('Data dir:', DATA_DIR.resolve())

Project dir: /content
Data dir: /content/data


## 1. Создание локальных модулей в Colab

Эта ячейка записывает код парсера и label-free модели в файловую систему Colab, чтобы дальше их можно было импортировать обычным способом.

In [11]:
from pathlib import Path

FILES = {
  "resume_parser/__init__.py": "\"\"\"resume parser package.\"\"\"\n\n__version__ = \"0.1.0\"\n",
  "resume_parser/html_tools.py": "from __future__ import annotations\n\nimport re\nfrom html import unescape\nfrom html.parser import HTMLParser\nfrom typing import Iterable, List\nfrom urllib.parse import urljoin, urlparse\n\n\nRESUME_PATH_RE = re.compile(r\"^/resume/[^/?#]+\\.html$\")\n\n\ndef normalize_space(text: str) -> str:\n    return re.sub(r\"\\s+\", \" \", unescape(text or \"\")).strip()\n\n\nclass ResumeLinkParser(HTMLParser):\n    def __init__(self, base_url: str) -> None:\n        super().__init__(convert_charrefs=True)\n        self.base_url = base_url\n        self.links: List[str] = []\n\n    def handle_starttag(self, tag: str, attrs: Iterable[tuple[str, str | None]]) -> None:\n        if tag.lower() != \"a\":\n            return\n\n        attrs_dict = dict(attrs)\n        href = attrs_dict.get(\"href\")\n        if not href:\n            return\n\n        absolute_url = urljoin(self.base_url, href)\n        parsed = urlparse(absolute_url)\n        if parsed.scheme not in {\"http\", \"https\"}:\n            return\n        if RESUME_PATH_RE.match(parsed.path) and absolute_url not in self.links:\n            self.links.append(absolute_url)\n\n\ndef extract_resume_links(html: str, base_url: str) -> List[str]:\n    parser = ResumeLinkParser(base_url)\n    parser.feed(html)\n    return parser.links\n\n\nBLOCK_TAGS = {\n    \"address\",\n    \"article\",\n    \"aside\",\n    \"blockquote\",\n    \"br\",\n    \"dd\",\n    \"div\",\n    \"dl\",\n    \"dt\",\n    \"fieldset\",\n    \"figcaption\",\n    \"figure\",\n    \"footer\",\n    \"form\",\n    \"h1\",\n    \"h2\",\n    \"h3\",\n    \"h4\",\n    \"h5\",\n    \"h6\",\n    \"header\",\n    \"hr\",\n    \"li\",\n    \"main\",\n    \"nav\",\n    \"ol\",\n    \"p\",\n    \"pre\",\n    \"section\",\n    \"table\",\n    \"td\",\n    \"th\",\n    \"tr\",\n    \"ul\",\n}\n\nSKIP_TAGS = {\"script\", \"style\", \"noscript\", \"svg\", \"head\", \"template\"}\n\n\nclass VisibleTextParser(HTMLParser):\n    def __init__(self) -> None:\n        super().__init__(convert_charrefs=True)\n        self._chunks: List[str] = []\n        self._skip_depth = 0\n\n    def handle_starttag(self, tag: str, attrs: Iterable[tuple[str, str | None]]) -> None:\n        tag = tag.lower()\n        if tag in SKIP_TAGS:\n            self._skip_depth += 1\n            return\n        if self._skip_depth:\n            return\n        if tag in BLOCK_TAGS:\n            self._chunks.append(\"\\n\")\n\n    def handle_endtag(self, tag: str) -> None:\n        tag = tag.lower()\n        if tag in SKIP_TAGS and self._skip_depth:\n            self._skip_depth -= 1\n            return\n        if self._skip_depth:\n            return\n        if tag in BLOCK_TAGS:\n            self._chunks.append(\"\\n\")\n\n    def handle_data(self, data: str) -> None:\n        if self._skip_depth:\n            return\n        data = normalize_space(data)\n        if data:\n            self._chunks.append(data)\n            self._chunks.append(\" \")\n\n    def lines(self) -> List[str]:\n        text = \"\".join(self._chunks)\n        lines = [normalize_space(line) for line in text.splitlines()]\n        cleaned: List[str] = []\n        for line in lines:\n            if not line:\n                continue\n            if cleaned and cleaned[-1] == line:\n                continue\n            cleaned.append(line)\n        return cleaned\n\n\ndef extract_visible_lines(html: str) -> List[str]:\n    parser = VisibleTextParser()\n    parser.feed(html)\n    return parser.lines()\n\n\nclass FirstHeadingParser(HTMLParser):\n    def __init__(self) -> None:\n        super().__init__(convert_charrefs=True)\n        self._in_h1 = False\n        self._chunks: List[str] = []\n        self.heading: str | None = None\n\n    def handle_starttag(self, tag: str, attrs: Iterable[tuple[str, str | None]]) -> None:\n        if tag.lower() == \"h1\" and self.heading is None:\n            self._in_h1 = True\n\n    def handle_endtag(self, tag: str) -> None:\n        if tag.lower() == \"h1\" and self._in_h1:\n            self.heading = normalize_space(\" \".join(self._chunks))\n            self._in_h1 = False\n\n    def handle_data(self, data: str) -> None:\n        if self._in_h1:\n            data = normalize_space(data)\n            if data:\n                self._chunks.append(data)\n\n\ndef extract_first_h1(html: str) -> str | None:\n    parser = FirstHeadingParser()\n    parser.feed(html)\n    return parser.heading or None\n",
  "resume_parser/privacy.py": "from __future__ import annotations\n\nimport hashlib\nimport re\nfrom typing import Iterable, List\n\n\nEMAIL_RE = re.compile(r\"(?i)\\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\\.[A-Z]{2,}\\b\")\nURL_RE = re.compile(\n    r\"(?i)\\b(?:https?://|www\\.|t\\.me/|vk\\.com/|linkedin\\.com/|github\\.com/|facebook\\.com/|instagram\\.com/)\\S+\"\n)\nDATE_IN_PARENTHESES_RE = re.compile(\n    r\"\\(\\s*\\b\\d{1,2}\\s+\"\n    r\"(?:января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\"\n    r\"\\s+\\d{4}\\b\\s*\\)\",\n    re.IGNORECASE,\n)\nDATE_OF_BIRTH_LABEL_RE = re.compile(\n    r\"\\b(?:дата рождения|день рождения)\\s*:?\\s*\\d{1,2}\\s+\"\n    r\"(?:января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\"\n    r\"\\s+\\d{4}\\b\",\n    re.IGNORECASE,\n)\nPHONE_CANDIDATE_RE = re.compile(r\"(?<!\\d)(?:\\+?\\d[\\d\\s().-]{7,}\\d)(?!\\d)\")\n\nPERSONAL_LABELS = {\n    \"имя\",\n    \"фамилия\",\n    \"отчество\",\n    \"фио\",\n    \"полное имя\",\n    \"дата рождения\",\n    \"день рождения\",\n    \"телефон\",\n    \"мобильный телефон\",\n    \"email\",\n    \"e-mail\",\n    \"почта\",\n    \"электронная почта\",\n    \"контакты\",\n    \"адрес\",\n    \"точный адрес\",\n    \"skype\",\n    \"telegram\",\n    \"гражданство\",\n    \"пол\",\n    \"семейное положение\",\n    \"дети\",\n}\n\nCONTACT_HINTS = (\n    \"зарегистрируйтесь или войдите, чтобы увидеть контакты\",\n    \"контакты соискателя\",\n    \"сохранить в word\",\n    \"скачать резюме\",\n)\n\n\ndef make_resume_hash(source: str, salt: str = \"resume-parser\") -> str:\n    digest = hashlib.sha256(f\"{salt}:{source}\".encode(\"utf-8\")).hexdigest()\n    return digest[:20]\n\n\ndef age_bucket(age: int | None) -> str | None:\n    if age is None or age < 14 or age > 100:\n        return None\n    if age < 25:\n        return \"18-24\"\n    if age < 35:\n        return \"25-34\"\n    if age < 45:\n        return \"35-44\"\n    if age < 55:\n        return \"45-54\"\n    if age < 65:\n        return \"55-64\"\n    return \"65+\"\n\n\ndef _redact_phone(match: re.Match[str]) -> str:\n    digits = re.sub(r\"\\D\", \"\", match.group(0))\n    if len(digits) < 10:\n        return match.group(0)\n    return \"[PHONE_REDACTED]\"\n\n\ndef redact_direct_identifiers(text: str | None) -> str | None:\n    if text is None:\n        return None\n    redacted = EMAIL_RE.sub(\"[EMAIL_REDACTED]\", text)\n    redacted = URL_RE.sub(\"[URL_REDACTED]\", redacted)\n    redacted = PHONE_CANDIDATE_RE.sub(_redact_phone, redacted)\n    redacted = DATE_IN_PARENTHESES_RE.sub(\"([DATE_REDACTED])\", redacted)\n    redacted = DATE_OF_BIRTH_LABEL_RE.sub(\"[DATE_REDACTED]\", redacted)\n    return redacted\n\n\ndef _looks_like_person_name(line: str) -> bool:\n    parts = line.split()\n    if not 1 <= len(parts) <= 3:\n        return False\n    if len(line) > 80:\n        return False\n    return all(re.match(r\"^[A-ZА-ЯЁ][A-Za-zА-Яа-яЁё.'-]+$\", part) for part in parts)\n\n\ndef remove_personal_lines(lines: Iterable[str]) -> List[str]:\n    cleaned: List[str] = []\n    skip_next = False\n    for line in lines:\n        normalized = line.strip().lower().rstrip(\":\")\n        if skip_next:\n            skip_next = False\n            continue\n        if normalized in PERSONAL_LABELS:\n            skip_next = True\n            continue\n        if any(hint in normalized for hint in CONTACT_HINTS):\n            continue\n        if normalized == \"возраст\" and cleaned and _looks_like_person_name(cleaned[-1]):\n            cleaned.pop()\n        cleaned.append(line)\n    return cleaned\n\n\ndef sanitize_lines(lines: Iterable[str]) -> List[str]:\n    return [\n        redacted\n        for line in remove_personal_lines(lines)\n        if (redacted := redact_direct_identifiers(line))\n    ]\n",
  "resume_parser/parser.py": "from __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nfrom typing import Any, Dict, Iterable, List, Sequence\n\nfrom .html_tools import extract_first_h1, extract_visible_lines, normalize_space\nfrom .privacy import age_bucket, make_resume_hash, redact_direct_identifiers, sanitize_lines\n\n\nMONTHS = {\n    \"январь\": 1,\n    \"января\": 1,\n    \"февраль\": 2,\n    \"февраля\": 2,\n    \"март\": 3,\n    \"марта\": 3,\n    \"апрель\": 4,\n    \"апреля\": 4,\n    \"май\": 5,\n    \"мая\": 5,\n    \"июнь\": 6,\n    \"июня\": 6,\n    \"июль\": 7,\n    \"июля\": 7,\n    \"август\": 8,\n    \"августа\": 8,\n    \"сентябрь\": 9,\n    \"сентября\": 9,\n    \"октябрь\": 10,\n    \"октября\": 10,\n    \"ноябрь\": 11,\n    \"ноября\": 11,\n    \"декабрь\": 12,\n    \"декабря\": 12,\n}\n\nSECTION_ENDS = {\n    \"образование\",\n    \"курсы и тренинги\",\n    \"иностранные языки\",\n    \"водительские права и авто\",\n    \"водительские права\",\n    \"ключевые навыки\",\n    \"навыки\",\n    \"о себе\",\n    \"дополнительная информация\",\n    \"рекомендации\",\n    \"похожие резюме\",\n}\n\nFOOTER_HINTS = (\n    \"похожие резюме\",\n    \"смотрите также\",\n    \"популярные специализации\",\n    \"инструменты соискателя\",\n    \"инструменты работодателя\",\n)\n\n\n@dataclass(frozen=True)\nclass ParseOptions:\n    include_redacted_text: bool = False\n    keep_sensitive: bool = False\n    keep_source_url: bool = False\n\n\ndef parse_resume_html(html: str, source_url: str, options: ParseOptions | None = None) -> Dict[str, Any]:\n    options = options or ParseOptions()\n    raw_lines = _trim_to_resume_body(extract_visible_lines(html), extract_first_h1(html))\n    lines = sanitize_lines(raw_lines)\n\n    result: Dict[str, Any] = {\n        \"source\": {\n            \"site\": \"careerist.ru\",\n            \"resume_hash\": make_resume_hash(source_url),\n        },\n        \"fetched_at\": datetime.now(timezone.utc).isoformat(),\n        \"privacy\": {\n            \"direct_identifiers_removed\": True,\n            \"source_url_kept\": options.keep_source_url,\n            \"excluded_by_default\": [\n                \"full_name\",\n                \"contacts\",\n                \"date_of_birth\",\n                \"photo\",\n                \"download_url\",\n                \"source_numeric_id\",\n                \"citizenship\",\n                \"gender\",\n                \"family_status\",\n            ],\n        },\n        \"resume\": {},\n    }\n    if options.keep_source_url:\n        result[\"source\"][\"url\"] = source_url\n\n    resume: Dict[str, Any] = result[\"resume\"]\n    resume[\"title\"] = _clean_title(extract_first_h1(html)) or _infer_title(lines)\n    resume[\"salary\"] = _parse_salary(_first_salary_line(lines))\n    resume[\"updated_at\"] = _parse_update_date(lines)\n    resume[\"city\"] = _value_after(lines, \"Город\")\n    resume[\"relocation\"] = _first_matching(lines, (\"переезд\",))\n    resume[\"employment_type\"] = _value_after(lines, \"Тип занятости\")\n    resume[\"total_experience_months\"] = _parse_duration_months(_value_after(lines, \"Опыт работы\"))\n\n    age_years = _parse_age(_value_after(lines, \"Возраст\"))\n    resume[\"age_bucket\"] = age_bucket(age_years)\n    if options.keep_sensitive:\n        resume[\"age_years\"] = age_years\n        resume[\"citizenship\"] = _value_after(lines, \"Гражданство\")\n\n    resume[\"work_experience\"] = _parse_work_experience(_section(lines, \"Опыт работы\", SECTION_ENDS))\n    resume[\"education\"] = _parse_timeline_section(_section(lines, \"Образование\", SECTION_ENDS))\n    resume[\"courses\"] = _parse_timeline_section(_section(lines, \"Курсы и тренинги\", SECTION_ENDS))\n    resume[\"languages\"] = _parse_languages(_section(lines, \"Иностранные языки\", SECTION_ENDS))\n    resume[\"driver_license_categories\"] = _parse_driver_license(\n        _section_any(lines, (\"Водительские права и авто\", \"Водительские права\"), SECTION_ENDS)\n    )\n    resume[\"skills\"] = _parse_skills(_section_any(lines, (\"Ключевые навыки\", \"Навыки\"), SECTION_ENDS))\n    resume[\"about\"] = _parse_about(_section(lines, \"О себе\", SECTION_ENDS))\n\n    if options.include_redacted_text:\n        resume[\"redacted_text\"] = \"\\n\".join(lines)\n\n    _drop_empty(resume)\n    return result\n\n\ndef _drop_empty(value: Any) -> None:\n    if isinstance(value, dict):\n        for key in list(value.keys()):\n            item = value[key]\n            if isinstance(item, (dict, list)):\n                _drop_empty(item)\n            if item in (None, \"\", [], {}):\n                value.pop(key, None)\n    elif isinstance(value, list):\n        for item in value:\n            _drop_empty(item)\n\n\ndef _trim_to_resume_body(lines: Sequence[str], h1: str | None) -> List[str]:\n    if not lines:\n        return []\n    if h1:\n        for idx, line in enumerate(lines):\n            if normalize_space(line) == normalize_space(h1):\n                lines = lines[idx:]\n                break\n\n    trimmed: List[str] = []\n    for line in lines:\n        if any(hint in line.lower() for hint in FOOTER_HINTS):\n            break\n        trimmed.append(line)\n    return trimmed\n\n\ndef _clean_title(title: str | None) -> str | None:\n    if not title:\n        return None\n    title = normalize_space(title)\n    if title.lower().startswith(\"резюме - \"):\n        title = title.split(\" - \", 1)[1]\n    return redact_direct_identifiers(title)\n\n\ndef _infer_title(lines: Sequence[str]) -> str | None:\n    for line in lines[:12]:\n        lower = line.lower()\n        if lower.startswith(\"от \") or lower in {\"возраст\", \"город\", \"гражданство\"}:\n            continue\n        if _parse_salary(line):\n            continue\n        return redact_direct_identifiers(line)\n    return None\n\n\ndef _line_key(line: str) -> str:\n    return normalize_space(line).lower().rstrip(\":\")\n\n\ndef _value_after(lines: Sequence[str], label: str) -> str | None:\n    wanted = _line_key(label)\n    for idx, line in enumerate(lines):\n        if _line_key(line) == wanted:\n            for candidate in lines[idx + 1 :]:\n                if candidate:\n                    return redact_direct_identifiers(candidate)\n    return None\n\n\ndef _first_matching(lines: Sequence[str], prefixes: Iterable[str]) -> str | None:\n    prefixes = tuple(prefix.lower() for prefix in prefixes)\n    for line in lines:\n        if line.lower().startswith(prefixes):\n            return redact_direct_identifiers(line)\n    return None\n\n\ndef _first_salary_line(lines: Sequence[str]) -> str | None:\n    for line in lines[:20]:\n        if _parse_salary(line):\n            return line\n    return None\n\n\ndef _parse_salary(line: str | None) -> Dict[str, Any] | None:\n    if not line:\n        return None\n    if \"договор\" in line.lower():\n        return {\"amount\": None, \"currency\": None, \"raw\": line}\n    match = re.search(r\"(?P<amount>\\d[\\d\\s]{2,})\\s*(?P<currency>руб|₽|usd|eur|\\$|€)?\", line, re.I)\n    if not match:\n        return None\n    amount = int(re.sub(r\"\\D\", \"\", match.group(\"amount\")))\n    currency_raw = (match.group(\"currency\") or \"\").lower()\n    currency = {\n        \"руб\": \"RUB\",\n        \"₽\": \"RUB\",\n        \"usd\": \"USD\",\n        \"$\": \"USD\",\n        \"eur\": \"EUR\",\n        \"€\": \"EUR\",\n    }.get(currency_raw, currency_raw.upper() or None)\n    return {\"amount\": amount, \"currency\": currency, \"raw\": line}\n\n\ndef _parse_update_date(lines: Sequence[str]) -> str | None:\n    for line in lines[:30]:\n        match = re.match(r\"от\\s+(\\d{1,2})\\s+([А-Яа-яЁё]+)\\s+(\\d{4})\", line)\n        if match:\n            day = int(match.group(1))\n            month = MONTHS.get(match.group(2).lower())\n            year = int(match.group(3))\n            if month:\n                return f\"{year:04d}-{month:02d}-{day:02d}\"\n    return None\n\n\ndef _parse_age(line: str | None) -> int | None:\n    if not line:\n        return None\n    match = re.search(r\"\\b(\\d{2})\\s+(?:год|года|лет)\\b\", line)\n    return int(match.group(1)) if match else None\n\n\ndef _parse_duration_months(line: str | None) -> int | None:\n    if not line:\n        return None\n    years = 0\n    months = 0\n    year_match = re.search(r\"(\\d+)\\s*(?:год|года|лет)\\b\", line, re.I)\n    month_match = re.search(r\"(\\d+)\\s*(?:месяц|месяца|месяцев)\\b\", line, re.I)\n    if year_match:\n        years = int(year_match.group(1))\n    if month_match:\n        months = int(month_match.group(1))\n    if not year_match and not month_match:\n        return None\n    return years * 12 + months\n\n\ndef _section(lines: Sequence[str], start_label: str, end_labels: Iterable[str]) -> List[str]:\n    return _section_any(lines, (start_label,), end_labels)\n\n\ndef _section_any(lines: Sequence[str], start_labels: Iterable[str], end_labels: Iterable[str]) -> List[str]:\n    start_keys = {_line_key(label) for label in start_labels}\n    end_keys = {_line_key(label) for label in end_labels}\n    start_idx = None\n    for idx, line in enumerate(lines):\n        if _line_key(line) in start_keys:\n            start_idx = idx + 1\n            break\n    if start_idx is None:\n        return []\n    end_idx = len(lines)\n    for idx in range(start_idx, len(lines)):\n        if _line_key(lines[idx]) in end_keys:\n            end_idx = idx\n            break\n    return [line for line in lines[start_idx:end_idx] if line]\n\n\ndef _looks_like_period(line: str) -> bool:\n    month_names = \"|\".join(MONTHS.keys())\n    return bool(\n        re.search(rf\"\\b(?:{month_names})\\s+\\d{{4}}\\s*[-—]\\s*(?:продолжаю работать|(?:{month_names})\\s+\\d{{4}})\", line, re.I)\n        or re.search(r\"\\b\\d{4}\\s*[-—]\\s*(?:\\d{4}|н\\.в\\.|настоящее время)\\b\", line, re.I)\n    )\n\n\ndef _parse_period(line: str) -> Dict[str, str | bool | None]:\n    parts = re.split(r\"\\s*[-—]\\s*\", line, maxsplit=1)\n    start = _parse_month_year(parts[0]) if parts else None\n    end_raw = parts[1] if len(parts) > 1 else None\n    is_current = bool(end_raw and re.search(r\"продолжаю|н\\.в\\.|настоящее\", end_raw, re.I))\n    end = None if is_current else _parse_month_year(end_raw)\n    return {\"start\": start, \"end\": end, \"is_current\": is_current, \"raw\": line}\n\n\ndef _parse_month_year(text: str | None) -> str | None:\n    if not text:\n        return None\n    month_names = \"|\".join(MONTHS.keys())\n    match = re.search(rf\"\\b({month_names})\\s+(\\d{{4}})\\b\", text, re.I)\n    if match:\n        month = MONTHS[match.group(1).lower()]\n        return f\"{int(match.group(2)):04d}-{month:02d}\"\n    match = re.search(r\"\\b(\\d{4})\\b\", text)\n    if match:\n        return match.group(1)\n    return None\n\n\ndef _header_before(lines: Sequence[str], duration_idx: int) -> tuple[int, List[str]]:\n    header: List[str] = []\n    idx = duration_idx - 1\n    while idx >= 0 and len(header) < 3:\n        line = lines[idx]\n        if line.startswith((\"•\", \"-\", \"–\")) or _looks_like_period(line) or _parse_duration_months(line):\n            break\n        if len(line) > 180 and header:\n            break\n        header.append(line)\n        idx -= 1\n    header.reverse()\n    return duration_idx - len(header), header\n\n\ndef _parse_work_experience(lines: Sequence[str]) -> List[Dict[str, Any]]:\n    if not lines:\n        return []\n    work_lines = list(lines)\n    if _parse_duration_months(work_lines[0]):\n        work_lines = work_lines[1:]\n\n    duration_indices = [\n        idx\n        for idx in range(len(work_lines) - 1)\n        if _parse_duration_months(work_lines[idx]) is not None and _looks_like_period(work_lines[idx + 1])\n    ]\n    if not duration_indices:\n        return []\n\n    header_starts: Dict[int, int] = {}\n    headers: Dict[int, List[str]] = {}\n    for duration_idx in duration_indices:\n        header_start, header = _header_before(work_lines, duration_idx)\n        header_starts[duration_idx] = header_start\n        headers[duration_idx] = header\n\n    entries: List[Dict[str, Any]] = []\n    for order, duration_idx in enumerate(duration_indices):\n        header = headers[duration_idx]\n        next_header_start = header_starts[duration_indices[order + 1]] if order + 1 < len(duration_indices) else len(work_lines)\n        description_lines = work_lines[duration_idx + 2 : next_header_start]\n        entry = {\n            \"position\": redact_direct_identifiers(header[0]) if len(header) > 0 else None,\n            \"company\": redact_direct_identifiers(header[1]) if len(header) > 1 else None,\n            \"industry\": redact_direct_identifiers(header[2]) if len(header) > 2 else None,\n            \"duration_months\": _parse_duration_months(work_lines[duration_idx]),\n            \"period\": _parse_period(work_lines[duration_idx + 1]),\n            \"description\": [redact_direct_identifiers(line) for line in description_lines if line],\n        }\n        _drop_empty(entry)\n        entries.append(entry)\n    return entries\n\n\ndef _parse_timeline_section(lines: Sequence[str]) -> List[Dict[str, Any]]:\n    if not lines:\n        return []\n    period_indices = [idx for idx, line in enumerate(lines) if re.search(r\"\\b\\d{4}\\s*[-—]\\s*\\d{4}\\b\", line)]\n    if not period_indices:\n        return [{\"text\": [redact_direct_identifiers(line) for line in lines]}]\n\n    entries: List[Dict[str, Any]] = []\n    cursor = 0\n    for idx in period_indices:\n        block = [redact_direct_identifiers(line) for line in lines[cursor : idx + 1] if line]\n        cursor = idx + 1\n        if not block:\n            continue\n        period = block[-1]\n        details = block[:-1]\n        entry = {\n            \"name\": details[0] if details else None,\n            \"level\": details[1] if len(details) > 1 else None,\n            \"details\": details[2:] if len(details) > 2 else [],\n            \"period\": period,\n        }\n        _drop_empty(entry)\n        entries.append(entry)\n    if cursor < len(lines):\n        tail = [redact_direct_identifiers(line) for line in lines[cursor:] if line]\n        if tail:\n            entries.append({\"text\": tail})\n    return entries\n\n\ndef _parse_languages(lines: Sequence[str]) -> List[Dict[str, str]]:\n    languages: List[Dict[str, str]] = []\n    for line in lines:\n        match = re.match(r\"(.+?)\\s*[—-]\\s*(.+)\", line)\n        if match:\n            languages.append({\"language\": match.group(1).strip(), \"level\": match.group(2).strip()})\n    return languages\n\n\ndef _parse_driver_license(lines: Sequence[str]) -> List[str]:\n    joined = \" \".join(lines)\n    match = re.search(r\"категори[ияй]:?\\s*([A-ZА-ЯЁ,\\s]+)\", joined, re.I)\n    if not match:\n        return []\n    return [item.strip().upper() for item in re.split(r\"[,;\\s]+\", match.group(1)) if item.strip()]\n\n\ndef _parse_skills(lines: Sequence[str]) -> List[str]:\n    skills: List[str] = []\n    for line in lines:\n        for item in re.split(r\"[,;•|]\", line):\n            item = normalize_space(item)\n            if item and len(item) <= 80 and item.lower() not in {\"ключевые навыки\", \"навыки\"}:\n                skills.append(item)\n    return sorted(set(skills), key=skills.index)\n\n\ndef _parse_about(lines: Sequence[str]) -> str | None:\n    if not lines:\n        return None\n    return redact_direct_identifiers(\" \".join(lines))\n",
  "resume_parser/fetcher.py": "from __future__ import annotations\n\nimport time\nimport urllib.error\nimport urllib.request\nimport urllib.robotparser\nfrom dataclasses import dataclass\nfrom typing import Dict, Tuple\nfrom urllib.parse import urlparse\n\n\nDEFAULT_USER_AGENT = \"ResumeParser/0.1 (+https://careerist.ru; privacy-first research crawler)\"\n\n\n@dataclass\nclass HttpResponse:\n    url: str\n    content: bytes\n    content_type: str | None\n    charset: str | None\n\n    def text(self) -> str:\n        charset = self.charset or \"utf-8\"\n        try:\n            return self.content.decode(charset, errors=\"replace\")\n        except LookupError:\n            return self.content.decode(\"utf-8\", errors=\"replace\")\n\n\nclass RobotsDisallowedError(PermissionError):\n    pass\n\n\nclass HttpClient:\n    def __init__(\n        self,\n        user_agent: str = DEFAULT_USER_AGENT,\n        delay_seconds: float = 1.0,\n        timeout_seconds: float = 30.0,\n        respect_robots: bool = True,\n    ) -> None:\n        self.user_agent = user_agent\n        self.delay_seconds = delay_seconds\n        self.timeout_seconds = timeout_seconds\n        self.respect_robots = respect_robots\n        self._last_request_at: Dict[str, float] = {}\n        self._robots: Dict[str, urllib.robotparser.RobotFileParser] = {}\n\n    def fetch(self, url: str) -> HttpResponse:\n        if self.respect_robots and not self._can_fetch(url):\n            raise RobotsDisallowedError(f\"robots.txt disallows fetching {url}\")\n        self._throttle(url)\n        request = urllib.request.Request(url, headers={\"User-Agent\": self.user_agent})\n        with urllib.request.urlopen(request, timeout=self.timeout_seconds) as response:\n            content = response.read()\n            content_type = response.headers.get_content_type()\n            charset = response.headers.get_content_charset()\n            return HttpResponse(response.geturl(), content, content_type, charset)\n\n    def _throttle(self, url: str) -> None:\n        host = urlparse(url).netloc\n        if not host or self.delay_seconds <= 0:\n            return\n        last = self._last_request_at.get(host)\n        if last is not None:\n            wait_for = self.delay_seconds - (time.monotonic() - last)\n            if wait_for > 0:\n                time.sleep(wait_for)\n        self._last_request_at[host] = time.monotonic()\n\n    def _can_fetch(self, url: str) -> bool:\n        parsed = urlparse(url)\n        root = f\"{parsed.scheme}://{parsed.netloc}\"\n        parser = self._robots.get(root)\n        if parser is None:\n            parser = self._load_robots(root)\n            self._robots[root] = parser\n        return parser.can_fetch(self.user_agent, url)\n\n    def _load_robots(self, root: str) -> urllib.robotparser.RobotFileParser:\n        robots_url = f\"{root}/robots.txt\"\n        parser = urllib.robotparser.RobotFileParser()\n        parser.set_url(robots_url)\n        request = urllib.request.Request(robots_url, headers={\"User-Agent\": self.user_agent})\n        try:\n            with urllib.request.urlopen(request, timeout=self.timeout_seconds) as response:\n                raw = response.read().decode(response.headers.get_content_charset() or \"utf-8\", errors=\"replace\")\n        except (urllib.error.URLError, TimeoutError) as exc:\n            raise RuntimeError(f\"Could not load robots.txt from {robots_url}: {exc}\") from exc\n        parser.parse(raw.splitlines())\n        return parser\n",
  "retention_model/__init__.py": "\"\"\"Retention risk survival model for anonymized resume data.\"\"\"\n\n__version__ = \"0.1.0\"\n",
  "retention_model/features.py": "from __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport re\nimport statistics\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Sequence, Tuple\n\nTOKEN_RE = re.compile(r\"[a-zа-яё0-9+#.]+\", re.IGNORECASE)\nJOB_NUMERIC_FEATURES = 4\n\nNUMERIC_FEATURE_NAMES = [\n    \"salary_log\",\n    \"salary_missing\",\n    \"total_experience_months\",\n    \"job_count\",\n    \"avg_job_duration_months\",\n    \"median_job_duration_months\",\n    \"max_job_duration_months\",\n    \"min_job_duration_months\",\n    \"short_job_share_6m\",\n    \"short_job_share_12m\",\n    \"current_job_duration_months\",\n    \"career_span_months\",\n    \"avg_gap_months\",\n    \"max_gap_months\",\n    \"education_count\",\n    \"courses_count\",\n    \"languages_count\",\n    \"skills_count\",\n    \"about_length_log\",\n    \"relocation_restricted\",\n    \"full_time_requested\",\n    \"salary_per_experience_log\",\n]\n\n\n@dataclass\nclass FeatureBatch:\n    resume_hashes: List[str]\n    sequence: List[List[List[float]]]\n    lengths: List[int]\n    numeric: List[List[float]]\n    numeric_feature_names: List[str]\n    text_dim: int\n    max_seq_len: int\n\n    @property\n    def seq_input_dim(self) -> int:\n        if not self.sequence or not self.sequence[0]:\n            return self.text_dim + JOB_NUMERIC_FEATURES\n        return len(self.sequence[0][0])\n\n    def take(self, indices: Sequence[int]) -> \"FeatureBatch\":\n        return FeatureBatch(\n            resume_hashes=[self.resume_hashes[i] for i in indices],\n            sequence=[self.sequence[i] for i in indices],\n            lengths=[self.lengths[i] for i in indices],\n            numeric=[self.numeric[i] for i in indices],\n            numeric_feature_names=list(self.numeric_feature_names),\n            text_dim=self.text_dim,\n            max_seq_len=self.max_seq_len,\n        )\n\n\n@dataclass\nclass NumericScaler:\n    mean: List[float]\n    scale: List[float]\n\n    @classmethod\n    def fit(cls, values: Sequence[Sequence[float]]) -> \"NumericScaler\":\n        if not values:\n            raise ValueError(\"Cannot fit scaler on an empty matrix.\")\n        columns = list(zip(*values))\n        mean = [sum(column) / len(column) for column in columns]\n        scale = []\n        for column, center in zip(columns, mean):\n            variance = sum((value - center) ** 2 for value in column) / len(column)\n            std = math.sqrt(variance)\n            scale.append(std if std >= 1e-6 else 1.0)\n        return cls(mean=mean, scale=scale)\n\n    @classmethod\n    def from_dict(cls, payload: Dict[str, Any]) -> \"NumericScaler\":\n        return cls(\n            mean=[float(value) for value in payload[\"mean\"]],\n            scale=[float(value) for value in payload[\"scale\"]],\n        )\n\n    def transform(self, values: Sequence[Sequence[float]]) -> List[List[float]]:\n        return [\n            [(value - self.mean[index]) / self.scale[index] for index, value in enumerate(row)]\n            for row in values\n        ]\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\"mean\": list(self.mean), \"scale\": list(self.scale)}\n\n\ndef load_resume_records(path: str | Path) -> List[Dict[str, Any]]:\n    records: List[Dict[str, Any]] = []\n    with Path(path).open(\"r\", encoding=\"utf-8\") as fh:\n        for line_number, line in enumerate(fh, start=1):\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                records.append(json.loads(line))\n            except json.JSONDecodeError as exc:\n                raise ValueError(f\"Invalid JSONL at {path}:{line_number}: {exc}\") from exc\n    return records\n\n\ndef load_labels(path: str | Path) -> Dict[str, Tuple[float, int]]:\n    labels: Dict[str, Tuple[float, int]] = {}\n    with Path(path).open(\"r\", encoding=\"utf-8\", newline=\"\") as fh:\n        reader = csv.DictReader(fh)\n        if not reader.fieldnames:\n            raise ValueError(\"Labels CSV must contain a header row.\")\n        for row_number, row in enumerate(reader, start=2):\n            resume_hash = (row.get(\"resume_hash\") or \"\").strip()\n            if not resume_hash:\n                raise ValueError(f\"Missing resume_hash at labels row {row_number}.\")\n            duration = _pick_float(\n                row,\n                \"duration_months\",\n                \"tenure_months\",\n                \"retention_months\",\n                \"time_to_event_months\",\n                \"time\",\n            )\n            event = _pick_bool(row, \"event_observed\", \"event\", \"left\", \"resigned\", \"fired\", \"early_exit\")\n            if duration is None:\n                raise ValueError(f\"Missing duration/tenure in months at labels row {row_number}.\")\n            if event is None:\n                raise ValueError(f\"Missing event_observed/event flag at labels row {row_number}.\")\n            if duration <= 0:\n                raise ValueError(f\"Duration must be positive at labels row {row_number}.\")\n            labels[resume_hash] = (float(duration), int(event))\n    return labels\n\n\ndef featurize_records(\n    records: Sequence[Dict[str, Any]],\n    *,\n    text_dim: int = 384,\n    max_seq_len: int = 12,\n) -> FeatureBatch:\n    if text_dim < 16:\n        raise ValueError(\"text_dim must be at least 16.\")\n    if max_seq_len < 1:\n        raise ValueError(\"max_seq_len must be at least 1.\")\n\n    sequence = [\n        [[0.0 for _ in range(text_dim + JOB_NUMERIC_FEATURES)] for _ in range(max_seq_len)]\n        for _ in records\n    ]\n    lengths = [1 for _ in records]\n    numeric = [[0.0 for _ in NUMERIC_FEATURE_NAMES] for _ in records]\n    resume_hashes: List[str] = []\n\n    for row_index, record in enumerate(records):\n        resume = record.get(\"resume\", {}) or {}\n        source = record.get(\"source\", {}) or {}\n        resume_hash = str(source.get(\"resume_hash\") or f\"row-{row_index}\")\n        resume_hashes.append(resume_hash)\n\n        jobs = _chronological_jobs(resume.get(\"work_experience\", []) or [])\n        if len(jobs) > max_seq_len:\n            jobs = jobs[-max_seq_len:]\n\n        if not jobs:\n            synthetic_job = {\n                \"position\": resume.get(\"title\") or \"\",\n                \"duration_months\": resume.get(\"total_experience_months\") or 0,\n                \"period\": {\"is_current\": False},\n                \"description\": [resume.get(\"about\") or \"\"],\n            }\n            jobs = [synthetic_job]\n\n        lengths[row_index] = max(1, min(len(jobs), max_seq_len))\n        for job_index, job in enumerate(jobs[:max_seq_len]):\n            sequence[row_index][job_index] = _job_vector(job, job_index, max_seq_len, text_dim)\n\n        numeric[row_index] = _numeric_features(resume)\n\n    return FeatureBatch(\n        resume_hashes=resume_hashes,\n        sequence=sequence,\n        lengths=lengths,\n        numeric=numeric,\n        numeric_feature_names=list(NUMERIC_FEATURE_NAMES),\n        text_dim=text_dim,\n        max_seq_len=max_seq_len,\n    )\n\n\ndef align_labels(batch: FeatureBatch, labels: Dict[str, Tuple[float, int]]) -> Tuple[FeatureBatch, List[float], List[float]]:\n    indices: List[int] = []\n    durations: List[float] = []\n    events: List[int] = []\n    for index, resume_hash in enumerate(batch.resume_hashes):\n        label = labels.get(resume_hash)\n        if label is None:\n            continue\n        duration, event = label\n        indices.append(index)\n        durations.append(duration)\n        events.append(event)\n\n    if not indices:\n        raise ValueError(\"No resume_hash values from JSONL matched the labels CSV.\")\n\n    return (\n        batch.take(indices),\n        durations,\n        [float(event) for event in events],\n    )\n\n\ndef hashed_text_vector(text: str, dim: int = 384) -> List[float]:\n    vector = [0.0 for _ in range(dim)]\n    tokens = TOKEN_RE.findall((text or \"\").lower())\n    if not tokens:\n        return vector\n\n    features = list(tokens)\n    features.extend(f\"{left}_{right}\" for left, right in zip(tokens, tokens[1:]))\n    for token in features:\n        digest = hashlib.blake2b(token.encode(\"utf-8\"), digest_size=8).digest()\n        bucket = int.from_bytes(digest[:4], \"little\") % dim\n        sign = 1.0 if digest[4] % 2 == 0 else -1.0\n        vector[bucket] += sign\n\n    norm = math.sqrt(sum(value * value for value in vector))\n    if norm > 0:\n        vector = [value / norm for value in vector]\n    return vector\n\n\ndef _job_vector(job: Dict[str, Any], job_index: int, max_seq_len: int, text_dim: int) -> List[float]:\n    description = job.get(\"description\") or []\n    if isinstance(description, list):\n        description_text = \" \".join(str(item) for item in description[:8])\n    else:\n        description_text = str(description)\n\n    text = \" \".join(\n        str(part)\n        for part in [\n            job.get(\"position\"),\n            job.get(\"industry\"),\n            description_text,\n        ]\n        if part\n    )\n    duration = _safe_float(job.get(\"duration_months\"))\n    period = job.get(\"period\") or {}\n    return hashed_text_vector(text, text_dim) + [\n        math.log1p(duration) / math.log(241.0),\n        1.0 if period.get(\"is_current\") else 0.0,\n        float(job_index + 1) / float(max_seq_len),\n        1.0 if description_text.strip() else 0.0,\n    ]\n\n\ndef _numeric_features(resume: Dict[str, Any]) -> List[float]:\n    jobs = _chronological_jobs(resume.get(\"work_experience\", []) or [])\n    durations = [_safe_float(job.get(\"duration_months\")) for job in jobs if _safe_float(job.get(\"duration_months\")) > 0]\n    if durations:\n        avg_duration = sum(durations) / len(durations)\n        median_duration = float(statistics.median(durations))\n        max_duration = max(durations)\n        min_duration = min(durations)\n        short_6 = sum(1 for duration in durations if duration < 6) / len(durations)\n        short_12 = sum(1 for duration in durations if duration < 12) / len(durations)\n    else:\n        avg_duration = median_duration = max_duration = min_duration = short_6 = short_12 = 0.0\n\n    current_duration = 0.0\n    for job in jobs:\n        if (job.get(\"period\") or {}).get(\"is_current\"):\n            current_duration = _safe_float(job.get(\"duration_months\"))\n            break\n\n    gaps, career_span = _career_gaps_and_span(jobs)\n    salary = resume.get(\"salary\") or {}\n    salary_amount = _safe_float(salary.get(\"amount\"))\n    total_experience = _safe_float(resume.get(\"total_experience_months\"))\n    about_length = len(str(resume.get(\"about\") or \"\"))\n    relocation = str(resume.get(\"relocation\") or \"\").lower()\n    employment = str(resume.get(\"employment_type\") or \"\").lower()\n    salary_per_exp = salary_amount / max(total_experience, 1.0)\n\n    return [\n        math.log1p(salary_amount),\n        1.0 if salary_amount <= 0 else 0.0,\n        total_experience,\n        float(len(jobs)),\n        avg_duration,\n        median_duration,\n        max_duration,\n        min_duration,\n        short_6,\n        short_12,\n        current_duration,\n        career_span,\n        sum(gaps) / len(gaps) if gaps else 0.0,\n        max(gaps) if gaps else 0.0,\n        float(len(resume.get(\"education\", []) or [])),\n        float(len(resume.get(\"courses\", []) or [])),\n        float(len(resume.get(\"languages\", []) or [])),\n        float(len(resume.get(\"skills\", []) or [])),\n        math.log1p(about_length),\n        1.0 if \"невозмож\" in relocation else 0.0,\n        1.0 if \"полная\" in employment else 0.0,\n        math.log1p(salary_per_exp),\n    ]\n\n\ndef _chronological_jobs(jobs: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    indexed = list(enumerate(jobs))\n    if not indexed:\n        return []\n\n    def key(item: Tuple[int, Dict[str, Any]]) -> Tuple[int, int]:\n        index, job = item\n        period = job.get(\"period\") or {}\n        start = _ym_to_month_index(period.get(\"start\"))\n        if start is None:\n            return (10**9, -index)\n        return (start, index)\n\n    return [job for _, job in sorted(indexed, key=key)]\n\n\ndef _career_gaps_and_span(jobs: Sequence[Dict[str, Any]]) -> Tuple[List[float], float]:\n    periods: List[Tuple[int, int]] = []\n    for job in jobs:\n        period = job.get(\"period\") or {}\n        start = _ym_to_month_index(period.get(\"start\"))\n        end = _ym_to_month_index(period.get(\"end\"))\n        duration = int(_safe_float(job.get(\"duration_months\")))\n        if start is None:\n            continue\n        if end is None:\n            end = start + max(duration, 1)\n        periods.append((start, max(end, start)))\n\n    if not periods:\n        return [], 0.0\n\n    periods.sort()\n    gaps: List[float] = []\n    previous_end = periods[0][1]\n    for start, end in periods[1:]:\n        gaps.append(float(max(0, start - previous_end)))\n        previous_end = max(previous_end, end)\n    span = float(max(end for _, end in periods) - min(start for start, _ in periods))\n    return gaps, span\n\n\ndef _ym_to_month_index(value: Any) -> int | None:\n    if value is None:\n        return None\n    text = str(value)\n    match = re.match(r\"^(\\d{4})(?:-(\\d{2}))?$\", text)\n    if not match:\n        return None\n    year = int(match.group(1))\n    month = int(match.group(2) or \"1\")\n    return year * 12 + month\n\n\ndef _safe_float(value: Any) -> float:\n    if value in (None, \"\"):\n        return 0.0\n    try:\n        return float(value)\n    except (TypeError, ValueError):\n        return 0.0\n\n\ndef _pick_float(row: Dict[str, str], *names: str) -> float | None:\n    for name in names:\n        value = row.get(name)\n        if value not in (None, \"\"):\n            try:\n                return float(str(value).replace(\",\", \".\"))\n            except ValueError:\n                return None\n    return None\n\n\ndef _pick_bool(row: Dict[str, str], *names: str) -> int | None:\n    true_values = {\"1\", \"true\", \"yes\", \"y\", \"да\", \"истина\", \"уволился\", \"left\", \"resigned\", \"fired\"}\n    false_values = {\"0\", \"false\", \"no\", \"n\", \"нет\", \"ложь\", \"работает\", \"active\", \"censored\"}\n    for name in names:\n        value = row.get(name)\n        if value in (None, \"\"):\n            continue\n        normalized = str(value).strip().lower()\n        if normalized in true_values:\n            return 1\n        if normalized in false_values:\n            return 0\n        try:\n            return 1 if float(normalized.replace(\",\", \".\")) > 0 else 0\n        except ValueError:\n            return None\n    return None\n",
  "retention_model/proxy_score.py": "from __future__ import annotations\n\nimport csv\nimport json\nimport math\nimport re\nfrom argparse import Namespace\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Sequence, Tuple\n\nfrom .features import load_resume_records\n\n\nTOKEN_RE = re.compile(r\"[a-zа-яё0-9+#.]+\", re.IGNORECASE)\n\n\n@dataclass\nclass ProxyRiskResult:\n    resume_hash: str\n    title: str | None\n    risk_score: float\n    risk_level: str\n    confidence: float\n    proxy_retention_6m: float\n    proxy_retention_12m: float\n    proxy_retention_24m: float\n    components: Dict[str, float]\n    drivers: List[str]\n    features: Dict[str, float]\n\n\ndef score_records(records: Sequence[Dict[str, Any]]) -> List[ProxyRiskResult]:\n    scored = [_score_record(record) for record in records]\n    if not scored:\n        return []\n\n    # Percentile correction makes the score useful for ranking within a parsed batch,\n    # while the base score still carries the absolute heuristic signal.\n    if len(scored) >= 5:\n        order = sorted(range(len(scored)), key=lambda index: scored[index].risk_score)\n        percentiles = [0.0 for _ in scored]\n        denominator = max(len(scored) - 1, 1)\n        for rank, index in enumerate(order):\n            percentiles[index] = rank / denominator\n    else:\n        percentiles = [item.risk_score for item in scored]\n\n    adjusted: List[ProxyRiskResult] = []\n    for item, percentile in zip(scored, percentiles):\n        blended = _clamp(0.72 * item.risk_score + 0.28 * percentile)\n        adjusted.append(\n            ProxyRiskResult(\n                resume_hash=item.resume_hash,\n                title=item.title,\n                risk_score=blended,\n                risk_level=_risk_level(blended),\n                confidence=item.confidence,\n                proxy_retention_6m=_retention_probability(blended, 6),\n                proxy_retention_12m=_retention_probability(blended, 12),\n                proxy_retention_24m=_retention_probability(blended, 24),\n                components=item.components,\n                drivers=item.drivers,\n                features=item.features,\n            )\n        )\n    return sorted(adjusted, key=lambda item: item.risk_score, reverse=True)\n\n\ndef score_cli(args: Namespace) -> None:\n    records = load_resume_records(args.resumes_jsonl)\n    scored = score_records(records)\n    out_path = Path(args.out)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    if args.format == \"jsonl\":\n        _write_jsonl(out_path, scored)\n    else:\n        _write_csv(out_path, scored)\n    print(f\"scored {len(scored)} resumes: {out_path}\")\n\n\ndef _score_record(record: Dict[str, Any]) -> ProxyRiskResult:\n    source = record.get(\"source\", {}) or {}\n    resume = record.get(\"resume\", {}) or {}\n    resume_hash = str(source.get(\"resume_hash\") or \"\")\n    title = resume.get(\"title\")\n    jobs = _chronological_jobs(resume.get(\"work_experience\", []) or [])\n    durations = [_safe_float(job.get(\"duration_months\")) for job in jobs if _safe_float(job.get(\"duration_months\")) > 0]\n\n    job_count = len(jobs)\n    total_experience = _safe_float(resume.get(\"total_experience_months\"))\n    if total_experience <= 0 and durations:\n        total_experience = sum(durations)\n\n    median_duration = _median(durations)\n    avg_duration = sum(durations) / len(durations) if durations else 0.0\n    max_duration = max(durations) if durations else 0.0\n    short_6_share = _share(durations, lambda value: value < 6)\n    short_12_share = _share(durations, lambda value: value < 12)\n    short_18_share = _share(durations, lambda value: value < 18)\n    switches_per_year = max(job_count - 1, 0) / max(total_experience / 12.0, 1.0)\n\n    latest_job = jobs[-1] if jobs else {}\n    latest_duration = _safe_float(latest_job.get(\"duration_months\"))\n    latest_is_current = bool((latest_job.get(\"period\") or {}).get(\"is_current\"))\n    gaps, career_span = _career_gaps_and_span(jobs)\n    avg_gap = sum(gaps) / len(gaps) if gaps else 0.0\n    max_gap = max(gaps) if gaps else 0.0\n    trajectory_similarity = _avg_transition_similarity(jobs)\n\n    components = {\n        \"short_tenure\": _clamp(0.45 * short_12_share + 0.35 * short_6_share + 0.20 * short_18_share),\n        \"switch_frequency\": _ramp(switches_per_year, 0.35, 1.15),\n        \"median_tenure\": 1.0 - _ramp(median_duration, 9, 36) if durations else 0.45,\n        \"recent_tenure\": _recent_tenure_risk(latest_duration, latest_is_current, bool(jobs)),\n        \"gaps\": _clamp(0.65 * _ramp(max_gap, 6, 18) + 0.35 * _ramp(avg_gap, 3, 12)),\n        \"trajectory_jumps\": 1.0 - trajectory_similarity if job_count >= 2 else 0.25,\n        \"data_scarcity\": 1.0 - _confidence(job_count, len(durations), total_experience, resume),\n    }\n    protective = _protective_signal(resume)\n    risk = (\n        0.26 * components[\"short_tenure\"]\n        + 0.21 * components[\"switch_frequency\"]\n        + 0.17 * components[\"median_tenure\"]\n        + 0.14 * components[\"recent_tenure\"]\n        + 0.08 * components[\"gaps\"]\n        + 0.06 * components[\"trajectory_jumps\"]\n        + 0.08 * components[\"data_scarcity\"]\n        - protective\n    )\n    risk = _clamp(risk)\n    confidence = _confidence(job_count, len(durations), total_experience, resume)\n    features = {\n        \"job_count\": float(job_count),\n        \"total_experience_months\": total_experience,\n        \"avg_job_duration_months\": avg_duration,\n        \"median_job_duration_months\": median_duration,\n        \"max_job_duration_months\": max_duration,\n        \"latest_job_duration_months\": latest_duration,\n        \"short_job_share_6m\": short_6_share,\n        \"short_job_share_12m\": short_12_share,\n        \"switches_per_year\": switches_per_year,\n        \"career_span_months\": career_span,\n        \"avg_gap_months\": avg_gap,\n        \"max_gap_months\": max_gap,\n        \"trajectory_similarity\": trajectory_similarity,\n        \"protective_signal\": protective,\n    }\n    return ProxyRiskResult(\n        resume_hash=resume_hash,\n        title=title,\n        risk_score=risk,\n        risk_level=_risk_level(risk),\n        confidence=confidence,\n        proxy_retention_6m=_retention_probability(risk, 6),\n        proxy_retention_12m=_retention_probability(risk, 12),\n        proxy_retention_24m=_retention_probability(risk, 24),\n        components=components,\n        drivers=_drivers(components, features, protective),\n        features=features,\n    )\n\n\ndef _write_csv(path: Path, scored: Sequence[ProxyRiskResult]) -> None:\n    fieldnames = [\n        \"resume_hash\",\n        \"title\",\n        \"risk_score\",\n        \"risk_percent\",\n        \"risk_level\",\n        \"confidence\",\n        \"proxy_retention_6m\",\n        \"proxy_retention_12m\",\n        \"proxy_retention_24m\",\n        \"drivers\",\n        \"job_count\",\n        \"total_experience_months\",\n        \"median_job_duration_months\",\n        \"short_job_share_12m\",\n        \"switches_per_year\",\n        \"max_gap_months\",\n        \"trajectory_similarity\",\n    ]\n    with path.open(\"w\", encoding=\"utf-8\", newline=\"\") as fh:\n        writer = csv.DictWriter(fh, fieldnames=fieldnames)\n        writer.writeheader()\n        for item in scored:\n            writer.writerow(\n                {\n                    \"resume_hash\": item.resume_hash,\n                    \"title\": item.title,\n                    \"risk_score\": round(item.risk_score, 4),\n                    \"risk_percent\": round(item.risk_score * 100, 1),\n                    \"risk_level\": item.risk_level,\n                    \"confidence\": round(item.confidence, 4),\n                    \"proxy_retention_6m\": round(item.proxy_retention_6m, 4),\n                    \"proxy_retention_12m\": round(item.proxy_retention_12m, 4),\n                    \"proxy_retention_24m\": round(item.proxy_retention_24m, 4),\n                    \"drivers\": \"; \".join(item.drivers),\n                    \"job_count\": int(item.features[\"job_count\"]),\n                    \"total_experience_months\": round(item.features[\"total_experience_months\"], 1),\n                    \"median_job_duration_months\": round(item.features[\"median_job_duration_months\"], 1),\n                    \"short_job_share_12m\": round(item.features[\"short_job_share_12m\"], 4),\n                    \"switches_per_year\": round(item.features[\"switches_per_year\"], 4),\n                    \"max_gap_months\": round(item.features[\"max_gap_months\"], 1),\n                    \"trajectory_similarity\": round(item.features[\"trajectory_similarity\"], 4),\n                }\n            )\n\n\ndef _write_jsonl(path: Path, scored: Sequence[ProxyRiskResult]) -> None:\n    with path.open(\"w\", encoding=\"utf-8\") as fh:\n        for item in scored:\n            payload = {\n                \"resume_hash\": item.resume_hash,\n                \"title\": item.title,\n                \"risk_score\": item.risk_score,\n                \"risk_level\": item.risk_level,\n                \"confidence\": item.confidence,\n                \"proxy_retention\": {\n                    \"6m\": item.proxy_retention_6m,\n                    \"12m\": item.proxy_retention_12m,\n                    \"24m\": item.proxy_retention_24m,\n                },\n                \"components\": item.components,\n                \"drivers\": item.drivers,\n                \"features\": item.features,\n                \"model_note\": \"Label-free proxy model based only on parsed resume career trajectory.\",\n            }\n            fh.write(json.dumps(payload, ensure_ascii=False) + \"\\n\")\n\n\ndef _retention_probability(risk: float, months: int) -> float:\n    # Heuristic conversion: high risk means steeper early-retention decay.\n    annual_exit_probability = 0.08 + 0.58 * _clamp(risk)\n    annual_hazard = -math.log(max(1.0 - annual_exit_probability, 1e-6))\n    return _clamp(math.exp(-annual_hazard * months / 12.0))\n\n\ndef _drivers(components: Dict[str, float], features: Dict[str, float], protective: float) -> List[str]:\n    labels = {\n        \"short_tenure\": \"много коротких мест работы\",\n        \"switch_frequency\": \"частая смена работодателей\",\n        \"median_tenure\": \"низкая медианная длительность работы\",\n        \"recent_tenure\": \"короткий текущий/последний период\",\n        \"gaps\": \"заметные перерывы между местами работы\",\n        \"trajectory_jumps\": \"слабая связность карьерных переходов\",\n        \"data_scarcity\": \"мало данных для надежной оценки\",\n    }\n    ranked = sorted(components.items(), key=lambda item: item[1], reverse=True)\n    drivers = [labels[key] for key, value in ranked if value >= 0.55][:4]\n    if protective >= 0.04:\n        drivers.append(\"есть компенсирующие сигналы: обучение/языки/полнота описания\")\n    if not drivers:\n        if features[\"median_job_duration_months\"] >= 24:\n            drivers.append(\"карьерная история выглядит устойчивой\")\n        else:\n            drivers.append(\"выраженных негативных сигналов немного\")\n    return drivers\n\n\ndef _protective_signal(resume: Dict[str, Any]) -> float:\n    courses = len(resume.get(\"courses\", []) or [])\n    languages = len(resume.get(\"languages\", []) or [])\n    skills = len(resume.get(\"skills\", []) or [])\n    about_len = len(str(resume.get(\"about\") or \"\"))\n    education = len(resume.get(\"education\", []) or [])\n    signal = (\n        0.02 * min(courses, 3)\n        + 0.015 * min(languages, 2)\n        + 0.015 * min(skills / 8.0, 1.0)\n        + 0.015 * min(about_len / 700.0, 1.0)\n        + 0.01 * min(education, 2)\n    )\n    return min(signal, 0.09)\n\n\ndef _confidence(job_count: int, known_durations: int, total_experience: float, resume: Dict[str, Any]) -> float:\n    confidence = 0.25\n    confidence += 0.20 * min(job_count / 3.0, 1.0)\n    confidence += 0.25 * min(known_durations / max(job_count, 1), 1.0)\n    confidence += 0.15 * min(total_experience / 36.0, 1.0)\n    if resume.get(\"about\"):\n        confidence += 0.05\n    if resume.get(\"education\"):\n        confidence += 0.05\n    if resume.get(\"salary\"):\n        confidence += 0.05\n    return _clamp(confidence)\n\n\ndef _recent_tenure_risk(duration: float, is_current: bool, has_jobs: bool) -> float:\n    if not has_jobs:\n        return 0.45\n    if duration <= 0:\n        return 0.40\n    base = 1.0 - _ramp(duration, 6, 30)\n    if is_current and duration >= 18:\n        base *= 0.65\n    return _clamp(base)\n\n\ndef _avg_transition_similarity(jobs: Sequence[Dict[str, Any]]) -> float:\n    if len(jobs) < 2:\n        return 0.75\n    scores: List[float] = []\n    for left, right in zip(jobs, jobs[1:]):\n        left_tokens = _job_tokens(left)\n        right_tokens = _job_tokens(right)\n        if not left_tokens or not right_tokens:\n            continue\n        scores.append(len(left_tokens & right_tokens) / len(left_tokens | right_tokens))\n    if not scores:\n        return 0.45\n    return _clamp(sum(scores) / len(scores))\n\n\ndef _job_tokens(job: Dict[str, Any]) -> set[str]:\n    text = \" \".join(str(job.get(key) or \"\") for key in (\"position\", \"industry\"))\n    return {token for token in TOKEN_RE.findall(text.lower()) if len(token) > 2}\n\n\ndef _chronological_jobs(jobs: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    indexed = list(enumerate(jobs))\n\n    def key(item: Tuple[int, Dict[str, Any]]) -> Tuple[int, int]:\n        index, job = item\n        start = _ym_to_month_index((job.get(\"period\") or {}).get(\"start\"))\n        if start is None:\n            return (10**9, index)\n        return (start, index)\n\n    return [job for _, job in sorted(indexed, key=key)]\n\n\ndef _career_gaps_and_span(jobs: Sequence[Dict[str, Any]]) -> Tuple[List[float], float]:\n    periods: List[Tuple[int, int]] = []\n    for job in jobs:\n        period = job.get(\"period\") or {}\n        start = _ym_to_month_index(period.get(\"start\"))\n        end = _ym_to_month_index(period.get(\"end\"))\n        duration = int(_safe_float(job.get(\"duration_months\")))\n        if start is None:\n            continue\n        if end is None:\n            end = start + max(duration, 1)\n        periods.append((start, max(end, start)))\n    if not periods:\n        return [], 0.0\n    periods.sort()\n    gaps: List[float] = []\n    previous_end = periods[0][1]\n    for start, end in periods[1:]:\n        gaps.append(float(max(0, start - previous_end)))\n        previous_end = max(previous_end, end)\n    return gaps, float(max(end for _, end in periods) - min(start for start, _ in periods))\n\n\ndef _ym_to_month_index(value: Any) -> int | None:\n    if value is None:\n        return None\n    match = re.match(r\"^(\\d{4})(?:-(\\d{2}))?$\", str(value))\n    if not match:\n        return None\n    return int(match.group(1)) * 12 + int(match.group(2) or \"1\")\n\n\ndef _median(values: Sequence[float]) -> float:\n    if not values:\n        return 0.0\n    ordered = sorted(values)\n    middle = len(ordered) // 2\n    if len(ordered) % 2:\n        return ordered[middle]\n    return (ordered[middle - 1] + ordered[middle]) / 2.0\n\n\ndef _share(values: Sequence[float], predicate: Any) -> float:\n    if not values:\n        return 0.0\n    return sum(1.0 for value in values if predicate(value)) / len(values)\n\n\ndef _safe_float(value: Any) -> float:\n    if value in (None, \"\"):\n        return 0.0\n    try:\n        return float(value)\n    except (TypeError, ValueError):\n        return 0.0\n\n\ndef _ramp(value: float, low: float, high: float) -> float:\n    if high <= low:\n        return 1.0 if value >= high else 0.0\n    return _clamp((value - low) / (high - low))\n\n\ndef _clamp(value: float, low: float = 0.0, high: float = 1.0) -> float:\n    return max(low, min(high, float(value)))\n\n\ndef _risk_level(score: float) -> str:\n    if score >= 0.68:\n        return \"high\"\n    if score >= 0.40:\n        return \"medium\"\n    return \"low\"\n",
  "retention_model/proxy_metrics.py": "from __future__ import annotations\n\nimport math\nimport random\nfrom typing import Any, Dict, List, Sequence\n\nfrom .proxy_score import ProxyRiskResult, score_records\n\n\nSANITY_FEATURES = {\n    \"short_job_share_12m\": 1,\n    \"switches_per_year\": 1,\n    \"max_gap_months\": 1,\n    \"median_job_duration_months\": -1,\n    \"trajectory_similarity\": -1,\n    \"protective_signal\": -1,\n}\n\n\ndef evaluate_proxy_model(\n    records: Sequence[Dict[str, Any]],\n    *,\n    n_bootstrap: int = 100,\n    sample_fraction: float = 0.8,\n    seed: int = 42,\n) -> Dict[str, Any]:\n    \"\"\"Diagnostic metrics for a label-free risk model.\n\n    These are not supervised quality metrics. They check coverage, confidence,\n    directional sanity, and score stability under resampling.\n    \"\"\"\n\n    results = score_records(records)\n    if not results:\n        return {\n            \"summary\": {\"n_records\": 0, \"n_scored\": 0, \"score_coverage\": 0.0},\n            \"sanity_correlations\": [],\n            \"bootstrap_stability\": [],\n        }\n\n    summary = _summary(results, len(records))\n    sanity = _sanity_correlations(results)\n    stability = _bootstrap_stability(records, results, n_bootstrap, sample_fraction, seed)\n    summary[\"sanity_direction_pass_rate\"] = _mean(\n        1.0 if item[\"direction_ok\"] else 0.0\n        for item in sanity\n        if not math.isnan(item[\"spearman_r\"])\n    )\n    summary[\"bootstrap_rank_stability_mean\"] = _mean(\n        item[\"spearman_r\"] for item in stability if not math.isnan(item[\"spearman_r\"])\n    )\n    summary[\"bootstrap_rank_stability_p10\"] = _percentile(\n        [item[\"spearman_r\"] for item in stability if not math.isnan(item[\"spearman_r\"])],\n        0.10,\n    )\n    return {\n        \"summary\": summary,\n        \"sanity_correlations\": sanity,\n        \"bootstrap_stability\": stability,\n    }\n\n\ndef _summary(results: Sequence[ProxyRiskResult], n_records: int) -> Dict[str, float]:\n    scores = [item.risk_score for item in results]\n    confidence = [item.confidence for item in results]\n    levels = [item.risk_level for item in results]\n    return {\n        \"n_records\": float(n_records),\n        \"n_scored\": float(len(results)),\n        \"score_coverage\": len(results) / max(n_records, 1),\n        \"risk_mean\": _mean(scores),\n        \"risk_std\": _std(scores),\n        \"risk_p10\": _percentile(scores, 0.10),\n        \"risk_p50\": _percentile(scores, 0.50),\n        \"risk_p90\": _percentile(scores, 0.90),\n        \"confidence_mean\": _mean(confidence),\n        \"confidence_p10\": _percentile(confidence, 0.10),\n        \"low_confidence_share\": _mean(1.0 if item.confidence < 0.55 else 0.0 for item in results),\n        \"high_risk_share\": levels.count(\"high\") / len(levels),\n        \"medium_risk_share\": levels.count(\"medium\") / len(levels),\n        \"low_risk_share\": levels.count(\"low\") / len(levels),\n    }\n\n\ndef _sanity_correlations(results: Sequence[ProxyRiskResult]) -> List[Dict[str, Any]]:\n    risk = [item.risk_score for item in results]\n    rows: List[Dict[str, Any]] = []\n    for feature, expected_direction in SANITY_FEATURES.items():\n        values = [item.features.get(feature, float(\"nan\")) for item in results]\n        rho = spearman_correlation(values, risk)\n        if math.isnan(rho):\n            direction_ok = False\n        elif expected_direction > 0:\n            direction_ok = rho >= 0\n        else:\n            direction_ok = rho <= 0\n        rows.append(\n            {\n                \"feature\": feature,\n                \"expected_direction\": \"+\" if expected_direction > 0 else \"-\",\n                \"spearman_r\": rho,\n                \"direction_ok\": direction_ok,\n            }\n        )\n    return rows\n\n\ndef _bootstrap_stability(\n    records: Sequence[Dict[str, Any]],\n    base_results: Sequence[ProxyRiskResult],\n    n_bootstrap: int,\n    sample_fraction: float,\n    seed: int,\n) -> List[Dict[str, float]]:\n    if len(records) < 5 or n_bootstrap <= 0:\n        return []\n\n    rng = random.Random(seed)\n    base_by_hash = {item.resume_hash: item.risk_score for item in base_results}\n    sample_size = max(5, min(len(records), int(round(len(records) * sample_fraction))))\n    rows: List[Dict[str, float]] = []\n\n    for iteration in range(1, n_bootstrap + 1):\n        sample = rng.sample(list(records), sample_size)\n        sampled_results = score_records(sample)\n        full_scores: List[float] = []\n        sampled_scores: List[float] = []\n        for item in sampled_results:\n            if item.resume_hash in base_by_hash:\n                full_scores.append(base_by_hash[item.resume_hash])\n                sampled_scores.append(item.risk_score)\n        rows.append(\n            {\n                \"iteration\": float(iteration),\n                \"sample_size\": float(sample_size),\n                \"spearman_r\": spearman_correlation(full_scores, sampled_scores),\n            }\n        )\n    return rows\n\n\ndef spearman_correlation(left: Sequence[float], right: Sequence[float]) -> float:\n    pairs = [\n        (float(a), float(b))\n        for a, b in zip(left, right)\n        if not (math.isnan(float(a)) or math.isnan(float(b)))\n    ]\n    if len(pairs) < 3:\n        return float(\"nan\")\n    left_ranks = _ranks([a for a, _ in pairs])\n    right_ranks = _ranks([b for _, b in pairs])\n    return _pearson(left_ranks, right_ranks)\n\n\ndef _ranks(values: Sequence[float]) -> List[float]:\n    order = sorted(range(len(values)), key=lambda index: values[index])\n    ranks = [0.0 for _ in values]\n    cursor = 0\n    while cursor < len(order):\n        end = cursor\n        while end + 1 < len(order) and values[order[end + 1]] == values[order[cursor]]:\n            end += 1\n        average_rank = (cursor + end) / 2.0 + 1.0\n        for position in range(cursor, end + 1):\n            ranks[order[position]] = average_rank\n        cursor = end + 1\n    return ranks\n\n\ndef _pearson(left: Sequence[float], right: Sequence[float]) -> float:\n    left_mean = _mean(left)\n    right_mean = _mean(right)\n    numerator = sum((a - left_mean) * (b - right_mean) for a, b in zip(left, right))\n    left_var = sum((a - left_mean) ** 2 for a in left)\n    right_var = sum((b - right_mean) ** 2 for b in right)\n    denominator = math.sqrt(left_var * right_var)\n    if denominator <= 0:\n        return float(\"nan\")\n    return numerator / denominator\n\n\ndef _mean(values: Sequence[float] | Any) -> float:\n    values = list(values)\n    if not values:\n        return float(\"nan\")\n    return sum(float(value) for value in values) / len(values)\n\n\ndef _std(values: Sequence[float]) -> float:\n    values = list(values)\n    if len(values) < 2:\n        return 0.0\n    center = _mean(values)\n    return math.sqrt(sum((float(value) - center) ** 2 for value in values) / (len(values) - 1))\n\n\ndef _percentile(values: Sequence[float], q: float) -> float:\n    values = sorted(float(value) for value in values)\n    if not values:\n        return float(\"nan\")\n    q = max(0.0, min(1.0, q))\n    position = (len(values) - 1) * q\n    lower = int(math.floor(position))\n    upper = int(math.ceil(position))\n    if lower == upper:\n        return values[lower]\n    weight = position - lower\n    return values[lower] * (1 - weight) + values[upper] * weight\n"
}

for path, content in FILES.items():
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')

print('Written modules:', len(FILES))
print('Includes proxy metrics:', 'retention_model/proxy_metrics.py' in FILES)

Written modules: 9
Includes proxy metrics: True


## 2. Параметры парсинга

Для первого запуска оставьте небольшой лимит. Если сайт начинает отвечать ошибками или капчей, уменьшите `MAX_RESUMES` и увеличьте `DELAY_SECONDS`.

In [12]:
RUN_PARSING = True

# Быстрый smoke-test: поставьте True, чтобы спарсить только 50 резюме.
# Для финального EDA оставьте False: будет использована статистически осмысленная выборка 50000.
QUICK_TEST_MODE = False

# Статистически осмысленный минимум для оценки долей с 95% доверительным уровнем
# и погрешностью около ±3 п.п. при худшем случае p=0.5.
CONFIDENCE_Z = 1.96
MARGIN_ERROR = 0.03
STAT_SAMPLE_SIZE = math.ceil((CONFIDENCE_Z ** 2 * 0.5 * 0.5) / (MARGIN_ERROR ** 2))
TARGET_SAMPLE_SIZE = 20000
MAX_RESUMES = 50 if QUICK_TEST_MODE else max(TARGET_SAMPLE_SIZE, STAT_SAMPLE_SIZE)

RESUMES_PER_PAGE_ESTIMATE = 25
START_PAGE = 0
END_PAGE = math.ceil(MAX_RESUMES / RESUMES_PER_PAGE_ESTIMATE) - 1

# Если сайт начинает ограничивать запросы, увеличьте DELAY_SECONDS до 1.5-2.0.
DELAY_SECONDS = 1.0
TIMEOUT_SECONDS = 30

# Метрики стабильности прокси-модели: больше итераций = стабильнее оценка, но дольше расчет.
PROXY_METRICS_BOOTSTRAPS = 30 if not QUICK_TEST_MODE else 5
PROXY_METRICS_SAMPLE_FRACTION = 0.8

OUT_JSONL = DATA_DIR / 'resumes.jsonl'
OUT_FLAT_CSV = DATA_DIR / 'resumes_flat.csv'
OUT_RISK_CSV = DATA_DIR / 'proxy_retention_risk.csv'
OUT_METRICS_SUMMARY_CSV = DATA_DIR / 'proxy_model_metrics_summary.csv'
OUT_SANITY_CSV = DATA_DIR / 'proxy_model_sanity_correlations.csv'
OUT_STABILITY_CSV = DATA_DIR / 'proxy_model_bootstrap_stability.csv'

estimated_requests = (END_PAGE - START_PAGE + 1) + MAX_RESUMES
estimated_minutes = estimated_requests * DELAY_SECONDS / 60
print({
    'RUN_PARSING': RUN_PARSING,
    'QUICK_TEST_MODE': QUICK_TEST_MODE,
    'STAT_SAMPLE_SIZE': STAT_SAMPLE_SIZE,
    'MAX_RESUMES': MAX_RESUMES,
    'pages': f'{START_PAGE}..{END_PAGE}',
    'estimated_requests': estimated_requests,
    'estimated_runtime_min_lower_bound': round(estimated_minutes, 1),
    'OUT_JSONL': str(OUT_JSONL),
})

{'RUN_PARSING': True, 'QUICK_TEST_MODE': False, 'STAT_SAMPLE_SIZE': 1068, 'MAX_RESUMES': 20000, 'pages': '0..799', 'estimated_requests': 20800, 'estimated_runtime_min_lower_bound': 346.7, 'OUT_JSONL': 'data/resumes.jsonl'}


## 3. Парсинг

Сценарий: страница списка резюме → ссылки `/resume/*.html` → детальная HTML-страница → JSONL без прямых персональных данных.

In [ ]:
from urllib.parse import urlencode

from resume_parser.fetcher import HttpClient, RobotsDisallowedError
from resume_parser.html_tools import extract_resume_links
from resume_parser.parser import ParseOptions, parse_resume_html

DEFAULT_LIST_URL = 'https://careerist.ru/search/?category=resume&region=all'


def list_url(page: int) -> str:
    if page == 0:
        return DEFAULT_LIST_URL
    return 'https://careerist.ru/search/?' + urlencode({
        'category': 'resume',
        'region': 'all',
        'page': page,
    })

def dedupe(items):
    seen = set()
    result = []
    for item in items:
        if item in seen:
            continue
        seen.add(item)
        result.append(item)
    return result


def parse(start_page=0, end_page=0, max_resumes=25, delay_seconds=1.0, out_jsonl=OUT_JSONL):
    client = HttpClient(delay_seconds=delay_seconds, timeout_seconds=TIMEOUT_SECONDS, respect_robots=True)
    resume_urls = []

    for page in range(start_page, end_page + 1):
        url = list_url(page)
        print('Fetching list:', url)
        try:
            html = client.fetch(url).text()
        except RobotsDisallowedError as exc:
            print('robots skip:', exc)
            continue
        links = extract_resume_links(html, url)
        print(f'  found {len(links)} resume links')
        resume_urls.extend(links)

    resume_urls = dedupe(resume_urls)[:max_resumes]
    print('Unique resume URLs selected:', len(resume_urls))

    options = ParseOptions(
        include_redacted_text=False,
        keep_sensitive=False,
        keep_source_url=False,
    )
    parsed = []
    out_jsonl = Path(out_jsonl)
    out_jsonl.parent.mkdir(parents=True, exist_ok=True)

    with out_jsonl.open('w', encoding='utf-8') as fh:
        for url in tqdm(resume_urls, desc='Parsing resumes'):
            try:
                html = client.fetch(url).text()
                item = parse_resume_html(html, url, options)
                fh.write(json.dumps(item, ensure_ascii=False) + '\n')
                parsed.append(item)
            except RobotsDisallowedError as exc:
                print('robots skip:', exc)
            except Exception as exc:
                print('error:', url, exc)

    print(f'Parsed {len(parsed)} resumes -> {out_jsonl}')
    return parsed


if RUN_PARSING:
    records = parse(START_PAGE, END_PAGE, MAX_RESUMES, DELAY_SECONDS, OUT_JSONL)
else:
    records = []
    if OUT_JSONL.exists():
        records = [json.loads(line) for line in OUT_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('Loaded existing records:', len(records))

Fetching list: https://careerist.ru/search/?category=resume&region=all
  found 25 resume links
Fetching list: https://careerist.ru/search/?category=resume&region=all&page=1
  found 25 resume links
Fetching list: https://careerist.ru/search/?category=resume&region=all&page=2
  found 25 resume links
Fetching list: https://careerist.ru/search/?category=resume&region=all&page=3
  found 25 resume links
Fetching list: https://careerist.ru/search/?category=resume&region=all&page=4
  found 25 resume links
Fetching list: https://careerist.ru/search/?category=resume&region=all&page=5
  found 25 resume links
Fetching list: https://careerist.ru/search/?category=resume&region=all&page=6


## 4. Загрузка и нормализация данных для EDA

Ниже строятся две таблицы:

- `resume_df` — одна строка на резюме;
- `experience_df` — одна строка на место работы.

In [ ]:
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Файл не найден: {path}. Сначала запустите парсинг или загрузите JSONL.')
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def ym_to_month_index(value):
    if not value:
        return None
    match = re.match(r'^(\d{4})(?:-(\d{2}))?$', str(value))
    if not match:
        return None
    return int(match.group(1)) * 12 + int(match.group(2) or '1')


def safe_float(value):
    if value in (None, ''):
        return float('nan')
    try:
        return float(value)
    except (TypeError, ValueError):
        return float('nan')


def career_gaps_and_span(jobs):
    periods = []
    for job in jobs or []:
        period = job.get('period') or {}
        start = ym_to_month_index(period.get('start'))
        end = ym_to_month_index(period.get('end'))
        duration = safe_float(job.get('duration_months'))
        if start is None:
            continue
        if end is None and not math.isnan(duration):
            end = start + max(int(duration), 1)
        if end is None:
            continue
        periods.append((start, max(end, start)))
    if not periods:
        return [], float('nan')
    periods.sort()
    gaps = []
    previous_end = periods[0][1]
    for start, end in periods[1:]:
        gaps.append(max(0, start - previous_end))
        previous_end = max(previous_end, end)
    span = max(end for _, end in periods) - min(start for start, _ in periods)
    return gaps, span


def flatten_resume_record(record):
    resume = record.get('resume', {}) or {}
    source = record.get('source', {}) or {}
    jobs = resume.get('work_experience', []) or []
    durations = [safe_float(job.get('duration_months')) for job in jobs]
    durations = [x for x in durations if not math.isnan(x) and x > 0]
    gaps, career_span = career_gaps_and_span(jobs)
    salary = resume.get('salary') or {}
    about = resume.get('about') or ''
    skills = resume.get('skills') or []
    languages = resume.get('languages') or []
    education = resume.get('education') or []
    courses = resume.get('courses') or []
    current_durations = [
        safe_float(job.get('duration_months'))
        for job in jobs
        if (job.get('period') or {}).get('is_current')
    ]
    return {
        'resume_hash': source.get('resume_hash'),
        'title': resume.get('title'),
        'city': resume.get('city'),
        'updated_at': resume.get('updated_at'),
        'salary_amount': salary.get('amount'),
        'salary_currency': salary.get('currency'),
        'employment_type': resume.get('employment_type'),
        'relocation': resume.get('relocation'),
        'age_bucket': resume.get('age_bucket'),
        'total_experience_months': resume.get('total_experience_months'),
        'job_count': len(jobs),
        'avg_job_duration_months': sum(durations) / len(durations) if durations else float('nan'),
        'median_job_duration_months': pd.Series(durations).median() if durations else float('nan'),
        'max_job_duration_months': max(durations) if durations else float('nan'),
        'min_job_duration_months': min(durations) if durations else float('nan'),
        'short_job_share_6m': sum(d < 6 for d in durations) / len(durations) if durations else float('nan'),
        'short_job_share_12m': sum(d < 12 for d in durations) / len(durations) if durations else float('nan'),
        'current_job_duration_months': max([d for d in current_durations if not math.isnan(d)], default=float('nan')),
        'career_span_months': career_span,
        'avg_gap_months': sum(gaps) / len(gaps) if gaps else 0,
        'max_gap_months': max(gaps) if gaps else 0,
        'education_count': len(education),
        'courses_count': len(courses),
        'languages_count': len(languages),
        'skills_count': len(skills),
        'about_length': len(about),
        'has_salary': int(salary.get('amount') is not None),
        'has_about': int(bool(about)),
    }


def build_experience_rows(records):
    rows = []
    for record in records:
        resume_hash = (record.get('source') or {}).get('resume_hash')
        jobs = ((record.get('resume') or {}).get('work_experience') or [])
        for order, job in enumerate(jobs, start=1):
            period = job.get('period') or {}
            description = job.get('description') or []
            if isinstance(description, list):
                description_text = ' '.join(str(x) for x in description)
            else:
                description_text = str(description)
            rows.append({
                'resume_hash': resume_hash,
                'job_order': order,
                'position': job.get('position'),
                'company': job.get('company'),
                'industry': job.get('industry'),
                'duration_months': job.get('duration_months'),
                'period_start': period.get('start'),
                'period_end': period.get('end'),
                'is_current': bool(period.get('is_current')),
                'description_length': len(description_text),
            })
    return rows

records = load_jsonl(OUT_JSONL)
resume_df = pd.DataFrame([flatten_resume_record(record) for record in records])
experience_df = pd.DataFrame(build_experience_rows(records))

for col in ['salary_amount', 'total_experience_months', 'job_count', 'avg_job_duration_months', 'median_job_duration_months']:
    if col in resume_df:
        resume_df[col] = pd.to_numeric(resume_df[col], errors='coerce')
if not experience_df.empty:
    experience_df['duration_months'] = pd.to_numeric(experience_df['duration_months'], errors='coerce')

resume_df.to_csv(OUT_FLAT_CSV, index=False)
print('resume_df:', resume_df.shape, 'experience_df:', experience_df.shape)
display(resume_df.head())
display(experience_df.head())

## 5. EDA: качество данных и полнота

In [ ]:
summary = pd.DataFrame({
    'metric': [
        'resumes', 'unique_resume_hashes', 'duplicates', 'experience_rows',
        'with_salary', 'with_work_experience', 'with_education', 'with_languages', 'with_about'
    ],
    'value': [
        len(resume_df),
        resume_df['resume_hash'].nunique(dropna=True),
        int(resume_df['resume_hash'].duplicated().sum()),
        len(experience_df),
        int(resume_df['has_salary'].sum()),
        int((resume_df['job_count'] > 0).sum()),
        int((resume_df['education_count'] > 0).sum()),
        int((resume_df['languages_count'] > 0).sum()),
        int(resume_df['has_about'].sum()),
    ]
})
display(summary)

missing = (
    resume_df.isna().mean()
    .sort_values(ascending=False)
    .rename('missing_share')
    .reset_index()
    .rename(columns={'index': 'column'})
)
display(missing.head(30))

plt.figure(figsize=(10, 5))
sns.barplot(data=missing.head(20), y='column', x='missing_share', color='#4C78A8')
plt.title('Доля пропусков по полям резюме')
plt.xlabel('Доля пропусков')
plt.ylabel('Поле')
plt.xlim(0, 1)
plt.show()

## 6. EDA: категориальные признаки

In [ ]:
def top_bar(df, column, n=15, title=None):
    if column not in df or df[column].dropna().empty:
        print(f'Нет данных для {column}')
        return
    data = df[column].fillna('Не указано').value_counts().head(n).reset_index()
    data.columns = [column, 'count']
    plt.figure(figsize=(10, max(4, 0.35 * len(data))))
    sns.barplot(data=data, y=column, x='count', color='#59A14F')
    plt.title(title or f'Top {n}: {column}')
    plt.xlabel('Количество')
    plt.ylabel(column)
    plt.show()
    display(data)

for col, title in [
    ('city', 'Top городов'),
    ('title', 'Top должностей'),
    ('employment_type', 'Тип занятости'),
    ('age_bucket', 'Возрастные бакеты'),
    ('relocation', 'Готовность к переезду'),
]:
    top_bar(resume_df, col, n=15, title=title)

if not experience_df.empty:
    top_bar(experience_df, 'industry', n=20, title='Top отраслей в опыте работы')
    top_bar(experience_df, 'position', n=20, title='Top должностей в опыте работы')

## 7. EDA: числовые распределения

In [ ]:
def hist_numeric(df, column, title=None, bins=30, clip_q=None):
    if column not in df:
        print(f'Нет колонки {column}')
        return
    data = pd.to_numeric(df[column], errors='coerce').dropna()
    if data.empty:
        print(f'Нет числовых данных для {column}')
        return
    if clip_q is not None:
        hi = data.quantile(clip_q)
        data = data[data <= hi]
    plt.figure(figsize=(9, 4))
    sns.histplot(data, bins=bins, kde=True, color='#4C78A8')
    plt.title(title or column)
    plt.xlabel(column)
    plt.ylabel('Количество')
    plt.show()
    display(data.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame(column).T)

hist_numeric(resume_df, 'salary_amount', 'Распределение желаемой зарплаты, без верхнего 1%', clip_q=0.99)
hist_numeric(resume_df, 'total_experience_months', 'Общий опыт, месяцев', clip_q=0.99)
hist_numeric(resume_df, 'job_count', 'Количество мест работы')
hist_numeric(resume_df, 'median_job_duration_months', 'Медианная длительность работы, месяцев', clip_q=0.99)
hist_numeric(resume_df, 'short_job_share_12m', 'Доля мест работы короче 12 месяцев')
hist_numeric(resume_df, 'max_gap_months', 'Максимальный перерыв между работами, месяцев', clip_q=0.99)

if not experience_df.empty:
    hist_numeric(experience_df, 'duration_months', 'Длительность отдельных мест работы, месяцев', clip_q=0.99)

## 8. EDA: связи между признаками

In [ ]:
numeric_cols = [
    'salary_amount', 'total_experience_months', 'job_count', 'avg_job_duration_months',
    'median_job_duration_months', 'max_job_duration_months', 'short_job_share_6m',
    'short_job_share_12m', 'current_job_duration_months', 'career_span_months',
    'avg_gap_months', 'max_gap_months', 'education_count', 'courses_count',
    'languages_count', 'skills_count', 'about_length'
]
numeric_cols = [col for col in numeric_cols if col in resume_df]

corr_df = resume_df[numeric_cols].apply(pd.to_numeric, errors='coerce')
if len(corr_df.dropna(how='all')) > 1:
    plt.figure(figsize=(12, 9))
    sns.heatmap(corr_df.corr(numeric_only=True), cmap='vlag', center=0, linewidths=0.5)
    plt.title('Корреляции числовых признаков резюме')
    plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=resume_df,
    x='total_experience_months',
    y='salary_amount',
    hue='job_count' if 'job_count' in resume_df else None,
    alpha=0.75,
)
plt.yscale('log')
plt.title('Зарплатные ожидания vs общий опыт')
plt.xlabel('Общий опыт, месяцев')
plt.ylabel('Зарплата, log scale')
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=resume_df, x='job_count', y='median_job_duration_months', alpha=0.75)
plt.title('Количество мест работы vs медианная длительность')
plt.xlabel('Количество мест работы')
plt.ylabel('Медианная длительность, месяцев')
plt.show()

## 9. EDA: карьерная стабильность и потенциальные риск-сигналы

In [ ]:
risk_signal_cols = [
    'resume_hash', 'title', 'city', 'job_count', 'total_experience_months',
    'median_job_duration_months', 'short_job_share_12m', 'max_gap_months',
    'avg_gap_months', 'current_job_duration_months'
]
risk_signal_cols = [col for col in risk_signal_cols if col in resume_df]

candidate_flags = resume_df.copy()
candidate_flags['flag_many_short_jobs'] = candidate_flags['short_job_share_12m'].fillna(0) >= 0.5
candidate_flags['flag_high_switch_rate'] = (
    (candidate_flags['job_count'].fillna(0) - 1) / (candidate_flags['total_experience_months'].fillna(0).clip(lower=12) / 12)
) >= 1.0
candidate_flags['flag_large_gap'] = candidate_flags['max_gap_months'].fillna(0) >= 12
candidate_flags['flag_low_data'] = candidate_flags['job_count'].fillna(0) == 0

flag_cols = [col for col in candidate_flags.columns if col.startswith('flag_')]
flag_summary = candidate_flags[flag_cols].mean().sort_values(ascending=False).reset_index()
flag_summary.columns = ['flag', 'share']
display(flag_summary)

plt.figure(figsize=(8, 4))
sns.barplot(data=flag_summary, y='flag', x='share', color='#F28E2B')
plt.title('Доля резюме с риск-сигналами')
plt.xlabel('Доля')
plt.ylabel('Сигнал')
plt.xlim(0, 1)
plt.show()

display(
    candidate_flags.loc[candidate_flags[flag_cols].any(axis=1), risk_signal_cols + flag_cols]
    .sort_values(['flag_many_short_jobs', 'flag_high_switch_rate', 'flag_large_gap'], ascending=False)
    .head(30)
)

## 10. Label-free модель риска раннего увольнения

Модель не требует HRIS-меток. Это скоринг по прокси-сигналам карьерной траектории, поэтому результат нужно интерпретировать как **ранжирование и объяснение риска**, а не доказанный факт будущего увольнения.

In [ ]:
from retention_model.proxy_score import score_records

scored = score_records(records)
risk_rows = []
for item in scored:
    row = {
        'resume_hash': item.resume_hash,
        'risk_score': item.risk_score,
        'risk_percent': item.risk_score * 100,
        'risk_level': item.risk_level,
        'confidence': item.confidence,
        'proxy_retention_6m': item.proxy_retention_6m,
        'proxy_retention_12m': item.proxy_retention_12m,
        'proxy_retention_24m': item.proxy_retention_24m,
        'drivers': '; '.join(item.drivers),
    }
    row.update({f'component_{k}': v for k, v in item.components.items()})
    row.update({f'model_{k}': v for k, v in item.features.items()})
    risk_rows.append(row)

risk_df = pd.DataFrame(risk_rows)
risk_df.to_csv(OUT_RISK_CSV, index=False)
model_df = resume_df.merge(risk_df, on='resume_hash', how='left')

print('Risk output:', OUT_RISK_CSV)
display(risk_df.head(20))

## 11. Диагностические метрики модели

Без HRIS-меток нельзя посчитать accuracy, ROC-AUC или C-index по факту увольнения. Поэтому здесь используются диагностические метрики label-free модели:

- покрытие и confidence;
- доля low/medium/high риска;
- направленные sanity-check корреляции;
- bootstrap-стабильность ранжирования;
- decile-анализ риска и компонент.


In [ ]:
from retention_model.proxy_metrics import evaluate_proxy_model

proxy_metrics = evaluate_proxy_model(
    records,
    n_bootstrap=PROXY_METRICS_BOOTSTRAPS,
    sample_fraction=PROXY_METRICS_SAMPLE_FRACTION,
    seed=42,
)

metrics_summary_df = pd.DataFrame([
    {'metric': key, 'value': value}
    for key, value in proxy_metrics['summary'].items()
])
sanity_df = pd.DataFrame(proxy_metrics['sanity_correlations'])
stability_df = pd.DataFrame(proxy_metrics['bootstrap_stability'])

metrics_summary_df.to_csv(OUT_METRICS_SUMMARY_CSV, index=False)
sanity_df.to_csv(OUT_SANITY_CSV, index=False)
stability_df.to_csv(OUT_STABILITY_CSV, index=False)

print('Proxy model metrics summary')
display(metrics_summary_df)
print('Directional sanity correlations')
display(sanity_df)
print('Bootstrap rank stability')
display(stability_df.describe() if not stability_df.empty else pd.DataFrame({'note': ['Недостаточно данных для bootstrap stability']}))

## 12. Визуализация метрик модели


In [ ]:
if not risk_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    key_metrics = metrics_summary_df[metrics_summary_df['metric'].isin([
        'score_coverage', 'confidence_mean', 'low_confidence_share',
        'high_risk_share', 'medium_risk_share', 'low_risk_share',
        'sanity_direction_pass_rate', 'bootstrap_rank_stability_mean'
    ])].copy()
    sns.barplot(data=key_metrics, y='metric', x='value', ax=axes[0], color='#4C78A8')
    axes[0].set_title('Ключевые диагностические метрики')
    axes[0].set_xlim(0, 1)
    axes[0].set_xlabel('Значение')
    axes[0].set_ylabel('')

    if not sanity_df.empty:
        sanity_plot = sanity_df.copy()
        sanity_plot['expected'] = sanity_plot['expected_direction'].map({'+': 'положительная', '-': 'отрицательная'})
        sns.barplot(data=sanity_plot, y='feature', x='spearman_r', hue='direction_ok', ax=axes[1], palette='Set2')
        axes[1].axvline(0, color='black', linewidth=1)
        axes[1].set_title('Sanity-check корреляции с risk_score')
        axes[1].set_xlabel('Spearman r')
        axes[1].set_ylabel('')
    else:
        axes[1].axis('off')

    if not stability_df.empty:
        sns.histplot(stability_df['spearman_r'].dropna(), bins=15, kde=True, ax=axes[2], color='#59A14F')
        axes[2].axvline(stability_df['spearman_r'].mean(), color='black', linestyle='--', label='mean')
        axes[2].set_title('Bootstrap-стабильность ранжирования')
        axes[2].set_xlabel('Spearman r между full-score и bootstrap-score')
        axes[2].legend()
    else:
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    decile_df = risk_df.copy()
    decile_df['risk_decile'] = pd.qcut(
        decile_df['risk_score'].rank(method='first'),
        q=min(10, len(decile_df)),
        labels=False,
        duplicates='drop'
    ) + 1

    decile_summary = decile_df.groupby('risk_decile').agg(
        risk_score=('risk_score', 'mean'),
        confidence=('confidence', 'mean'),
        retention_6m=('proxy_retention_6m', 'mean'),
        retention_12m=('proxy_retention_12m', 'mean'),
        retention_24m=('proxy_retention_24m', 'mean'),
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.lineplot(data=decile_summary, x='risk_decile', y='risk_score', marker='o', ax=axes[0], label='risk_score')
    sns.lineplot(data=decile_summary, x='risk_decile', y='confidence', marker='o', ax=axes[0], label='confidence')
    axes[0].set_title('Risk/confidence по децилям риска')
    axes[0].set_xlabel('Дециль риска')
    axes[0].set_ylabel('Среднее значение')
    axes[0].set_ylim(0, 1)

    retention_long = decile_summary.melt(
        id_vars='risk_decile',
        value_vars=['retention_6m', 'retention_12m', 'retention_24m'],
        var_name='horizon',
        value_name='proxy_retention'
    )
    sns.lineplot(data=retention_long, x='risk_decile', y='proxy_retention', hue='horizon', marker='o', ax=axes[1])
    axes[1].set_title('Прокси-удержание по децилям риска')
    axes[1].set_xlabel('Дециль риска')
    axes[1].set_ylabel('Proxy retention')
    axes[1].set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

    component_cols = [col for col in risk_df.columns if col.startswith('component_')]
    if component_cols:
        component_heat = decile_df.groupby('risk_decile')[component_cols].mean()
        plt.figure(figsize=(12, 6))
        sns.heatmap(component_heat.T, cmap='YlOrRd', vmin=0, vmax=1, linewidths=0.5)
        plt.title('Компоненты риска по децилям')
        plt.xlabel('Дециль риска')
        plt.ylabel('Компонента')
        plt.show()

    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=risk_df, x='confidence', y='risk_score', hue='risk_level', alpha=0.8, palette='Set2')
    plt.axvline(0.55, color='gray', linestyle='--', linewidth=1)
    plt.axhline(0.68, color='red', linestyle='--', linewidth=1)
    plt.title('Матрица риск / уверенность')
    plt.xlabel('confidence')
    plt.ylabel('risk_score')
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.show()

## 13. EDA по результатам модели

In [ ]:
if not risk_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    sns.histplot(risk_df['risk_score'], bins=20, kde=True, ax=axes[0], color='#E15759')
    axes[0].set_title('Распределение risk_score')
    axes[0].set_xlabel('risk_score')

    sns.countplot(data=risk_df, x='risk_level', order=['low', 'medium', 'high'], ax=axes[1], palette='Set2')
    axes[1].set_title('Количество по risk_level')
    axes[1].set_xlabel('risk_level')

    sns.histplot(risk_df['confidence'], bins=20, kde=True, ax=axes[2], color='#4C78A8')
    axes[2].set_title('Confidence модели')
    axes[2].set_xlabel('confidence')
    plt.tight_layout()
    plt.show()

    driver_counts = (
        risk_df['drivers']
        .fillna('')
        .str.split('; ')
        .explode()
        .replace('', pd.NA)
        .dropna()
        .value_counts()
        .head(15)
        .reset_index()
    )
    driver_counts.columns = ['driver', 'count']
    plt.figure(figsize=(10, max(4, 0.35 * len(driver_counts))))
    sns.barplot(data=driver_counts, y='driver', x='count', color='#B07AA1')
    plt.title('Главные причины риска')
    plt.xlabel('Количество')
    plt.ylabel('Драйвер')
    plt.show()
    display(driver_counts)

    component_cols = [col for col in risk_df.columns if col.startswith('component_')]
    component_means = risk_df[component_cols].mean().sort_values(ascending=False).reset_index()
    component_means.columns = ['component', 'mean_value']
    plt.figure(figsize=(9, 4))
    sns.barplot(data=component_means, y='component', x='mean_value', color='#FF9DA7')
    plt.title('Средний вклад компонент риска')
    plt.xlabel('Среднее значение компоненты')
    plt.ylabel('Компонента')
    plt.xlim(0, 1)
    plt.show()
    display(component_means)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.scatterplot(data=model_df, x='median_job_duration_months', y='risk_score', hue='risk_level', ax=axes[0], alpha=0.8)
    axes[0].set_title('Risk vs медианная длительность работы')
    axes[0].set_xlabel('Медианная длительность, месяцев')
    axes[0].set_ylabel('risk_score')

    sns.scatterplot(data=model_df, x='short_job_share_12m', y='risk_score', hue='risk_level', ax=axes[1], alpha=0.8)
    axes[1].set_title('Risk vs доля коротких мест')
    axes[1].set_xlabel('Доля мест < 12 месяцев')
    axes[1].set_ylabel('risk_score')
    plt.tight_layout()
    plt.show()

    display(
        model_df[[
            'resume_hash', 'title', 'city', 'risk_percent', 'risk_level', 'confidence',
            'proxy_retention_12m', 'drivers', 'job_count', 'median_job_duration_months',
            'short_job_share_12m', 'max_gap_months'
        ]].sort_values('risk_percent', ascending=False).head(25)
    )

## 14. Сохранение и скачивание результатов

Файлы сохраняются в `data/`:

- `resumes.jsonl` — структурированные резюме без прямых персональных данных;
- `resumes_flat.csv` — плоская таблица для EDA;
- `proxy_retention_risk.csv` — результаты label-free модели.

In [ ]:
print('Saved files:')
for path in [
    OUT_JSONL,
    OUT_FLAT_CSV,
    OUT_RISK_CSV,
    OUT_METRICS_SUMMARY_CSV,
    OUT_SANITY_CSV,
    OUT_STABILITY_CSV,
]:
    path = Path(path)
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

# В Colab можно раскомментировать скачивание:
# from google.colab import files
# files.download(str(OUT_RISK_CSV))
# files.download(str(OUT_FLAT_CSV))
# files.download(str(OUT_METRICS_SUMMARY_CSV))